# Replicating Mouton (2021) Figures 3 and 4

Uses the steady-state Optical Bloch solver `ba138_spd_zeeman_linsolve_v2` to plot the
P-state population of $^{138}\mathrm{Ba}^+$ vs. the repump-laser detuning $\Delta_2$.

Conventions:
- B-field for both figures: 0.1 G = $10\,\mu\mathrm{T}$ (Mouton).
- Saturation Rabi frequencies derived from the script's decay rates: $\Omega_\mathrm{sat} = \gamma/\sqrt{2}$.
- Polarisations after Dijck et al. 2015: $\lambda_1$ linear $\parallel B$ (pure $\pi$), $\lambda_2$ circular and propagating $\perp B$, decomposing onto $(\sigma^-, \pi, \sigma^+) = (\tfrac12, \tfrac{1}{\sqrt2}, \tfrac12)$ in the B-frame.
- Laser-linewidth dephasing $\gamma_l$ available in the model (Dijck §R-matrix), but defaults to **0** here so the two-photon (EIT) dip reaches $\rho_{PP}=0$ at $\Delta_1 = \Delta_2 = 0$ as in Mouton's idealised plot. Set `GAMMA_L_MHZ > 0` (e.g. 1 MHz) to model a realistic lab linewidth — this lifts the dip floor.
- Zeeman shifts in the script were corrected to use $\hbar$ (angular convention) so they share units with $\Delta$, $\Omega$, $\gamma$.
- X-axis offset by 880 MHz to match Mouton's $\nu_2 - 461\,\mathrm{THz}$ labels.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))
from ba138_spd_zeeman_levels_optical_bloch_linsolve_v2 import ba138_spd_zeeman_linsolve_v2

In [ ]:
GAMMA_S = 95.3   # rad*MHz, S->P decay (script convention; Gamma_S ~ 15.17 MHz)
GAMMA_D = 31.0   # rad*MHz, D->P decay (script convention; Gamma_D ~ 4.93 MHz)
# GAMMA_S = 2 * np.pi * 16.5   # rad/s, S->P decay
# GAMMA_D = 2 * np.pi * 2.3   # rad/s, S->P decay

OMEGA_S_SAT = GAMMA_S / np.sqrt(2)   # Omega_sat = 2*pi*Gamma/sqrt(2) (Mouton's saturation convention)
OMEGA_D_SAT = GAMMA_D / np.sqrt(2)

# Saturation-convention rescaling. Mouton himself flags (p.12) that his I/I_sat
# labels do not map onto Dijck's at face value. With OMEGA_*_SAT defined as above,
# matching Mouton Fig 3 needs:
#   * scale_S ~ 5  -> sets Autler-Townes splitting (peak position)
#   * scale_D ~ 1.5 -> sets the saturation curve (relative peak heights vs I_2)
# Both leave back-compat untouched when set to 1.0.
# OMEGA_S_SCALE = 3.0
# OMEGA_D_SCALE = 5.0
OMEGA_S_SCALE = 5.0
OMEGA_D_SCALE = 1.5


B_MOUTON = 10e-6                     # 0.1 Gauss

# Laser-linewidth dephasing (FWHM, MHz). Set to 0 to reproduce Mouton's idealised
# Fig 3 / Fig 4 (perfect EIT: rho_PP -> 0 at Delta_1 = Delta_2 = 0). Set to ~1 MHz
# to model a realistic lab linewidth (lifts the EIT dip floor by ~gamma_l/Gamma).
GAMMA_L_MHZ = 0.0
GAMMA_L = 2 * np.pi * GAMMA_L_MHZ

# Polarisations after Dijck et al. 2015. The script's theta args now accept either a
# scalar theta (linear at angle theta from B) or an explicit (sigma-, pi, sigma+)
# amplitude triple in the B-frame.
#
# Dijck: lambda_1 linearly polarised parallel to B  ->  pure pi.
# Dijck: lambda_2 circularly polarised, beam propagates perpendicular to B.
#   A circularly polarised plane wave propagating perpendicular to the
#   quantisation axis decomposes onto (sigma-, pi, sigma+) as (1/2, 1/sqrt(2), 1/2).
#   Pure (0, 0, 1) sigma+ would optically pump into dark high-m_J D states.
POL_S_DEFAULT = np.pi / 2                                  # linear pi || B
POL_D_DEFAULT = (0.5, 1.0 / np.sqrt(2), 0.5)               # circular w.r.t. y-axis propagation


def p_state_population(Delta_S_MHz, Delta_D_MHz, Omega_S, Omega_D,
                       pol_s=POL_S_DEFAULT, pol_d=POL_D_DEFAULT,
                       B=B_MOUTON, gamma_l=GAMMA_L):
    """Steady-state total P-state population rho_PP for given detunings/Rabi/pol/B/gamma_l."""
    Delta_SP01 = 2 * np.pi * Delta_S_MHz
    Delta_DP01 = 2 * np.pi * Delta_D_MHz
    sigma_end, cp0, cp0_c, cp1, cp1_c = ba138_spd_zeeman_linsolve_v2(
        Delta_SP01, Delta_DP01, Omega_S, Omega_D, pol_s, pol_d,
        B=B, gamma_l=gamma_l)
    return float(sigma_end[cp0 * cp0_c == 1].real
                 + sigma_end[cp1 * cp1_c == 1].real)


def scan_delta_D(Delta_S_MHz, Delta_D_array_MHz, Omega_S, Omega_D, **kwargs):
    """1D scan of P-state population over D-P detunings."""
    return np.array([p_state_population(Delta_S_MHz, dd, Omega_S, Omega_D, **kwargs)
                     for dd in Delta_D_array_MHz])

## Figure 3 — vary $I_2$
$\Delta_1 = 0$, $I_1 = I_{1,\mathrm{sat}}$, $B = 0.1\,\mathrm{G}$, $I_2/I_{2,\mathrm{sat}} \in \{0.03, 0.1, 0.2, 0.4\}$.

In [ ]:
delta_D_MHz = np.linspace(-50, 40, 401)
nu2_axis_MHz = delta_D_MHz + 880   # ν2 - 461 THz, MHz

I2_ratios = [0.03, 0.1, 0.2, 0.4]
Omega_S_fig3 = OMEGA_S_SCALE * OMEGA_S_SAT       # I1 = I1,sat (Mouton-rescaled)
Delta_S_fig3 = 0.0                                # Delta_1 = 0

fig, ax = plt.subplots(figsize=(7, 5))
for r in I2_ratios:
    Omega_D_val = OMEGA_D_SCALE * OMEGA_D_SAT * np.sqrt(r)   # I ~ |Omega|^2
    pop = scan_delta_D(Delta_S_fig3, delta_D_MHz, Omega_S_fig3, Omega_D_val)
    ax.plot(nu2_axis_MHz, pop, label=fr'$I_2 = {r:g}\, I_{{2,\mathrm{{sat}}}}$')

ax.set_xlabel(r'$\nu_2 - 461\,\mathrm{THz}$ in MHz')
ax.set_ylabel('Brightness (P-state population)')
ax.set_title('Figure 3 (Mouton): brightness vs $\\nu_2$ for varied $I_2$')
ax.legend()
plt.tight_layout()
plt.show()

## Figure 4 — vary $\Delta_1$
Both intensities at saturation, $B = 0.1\,\mathrm{G}$, $\Delta_1 \in \{0, -12, -24, -36\}\,\mathrm{MHz}$.

In [ ]:
delta_D_MHz = np.linspace(-50, 40, 401)
nu2_axis_MHz = delta_D_MHz + 880

Delta_1_values = [0, -12, -24, -36]
Omega_S_fig4 = OMEGA_S_SCALE * OMEGA_S_SAT
Omega_D_fig4 = OMEGA_D_SCALE * OMEGA_D_SAT

fig, ax = plt.subplots(figsize=(7, 5))
for d1 in Delta_1_values:
    pop = scan_delta_D(d1, delta_D_MHz, Omega_S_fig4, Omega_D_fig4)
    ax.plot(nu2_axis_MHz, pop, label=fr'$\Delta_1 = {d1}\,\mathrm{{MHz}}$')

ax.set_xlabel(r'$\nu_2 - 461\,\mathrm{THz}$ in MHz')
ax.set_ylabel('Brightness (P-state population)')
ax.set_title('Figure 4 (Mouton): brightness vs $\\nu_2$ for varied $\\Delta_1$')
ax.legend()
plt.tight_layout()
plt.show()